<a href="https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I use Logistic Regression as my primary model because this is a binary prediction and ranking problem, and Logistic Regression is simple, reproducible, and interpretable. I use a constrained Random Forest as a challenger because relationships among demand, CTR, position, and position instability may be nonlinear.

The models produce decline probabilities, which I use to rank pages for a fixed Top-20 human-review queue. The primary metric is Precision@20, matching my Week-4 operational baseline. Model complexity is accepted only when it produces a meaningful held-out improvement.

In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import sklearn

from IPython.display import Markdown, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 190)

# ---------------------------------------------------------
# Secure Hugging Face connection
# ---------------------------------------------------------

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

assert HF_TOKEN, (
    "HF_TOKEN is missing. Add a Hugging Face READ token "
    "to Colab Secrets with the name HF_TOKEN."
)

safe_token = HF_TOKEN.replace("'", "''")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet'"
    f")"
)

# ---------------------------------------------------------
# Rebuild the same frame used in Week 4
# ---------------------------------------------------------

feature_sql = f"""
WITH aggregated AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT report_date) FILTER (
            WHERE report_date < DATE '2026-03-22'
        ) AS hist_days,

        COUNT(DISTINCT report_date) FILTER (
            WHERE report_date >= DATE '2026-03-22'
        ) AS outcome_days,

        SUM(gsc_impressions) FILTER (
            WHERE report_date < DATE '2026-03-22'
        ) AS hist_impressions,

        SUM(gsc_clicks) FILTER (
            WHERE report_date < DATE '2026-03-22'
        ) AS hist_clicks,

        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '2026-03-22'
        ) AS outcome_impressions,

        (
            SUM(
                gsc_avg_position * gsc_impressions
            ) FILTER (
                WHERE report_date < DATE '2026-03-22'
            )
            /
            NULLIF(
                SUM(gsc_impressions) FILTER (
                    WHERE report_date < DATE '2026-03-22'
                ),
                0
            )
        ) AS hist_avg_position,

        STDDEV_SAMP(gsc_avg_position) FILTER (
            WHERE report_date < DATE '2026-03-22'
              AND gsc_impressions > 0
        ) AS hist_position_sd

    FROM {MARCH_DAILY}

    WHERE ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
),

framed AS (
    SELECT
        *,

        1.0 * hist_impressions
            / NULLIF(hist_days, 0)
            AS hist_imp_per_day,

        1.0 * hist_clicks
            / NULLIF(hist_days, 0)
            AS hist_click_per_day,

        1.0 * hist_clicks
            / NULLIF(hist_impressions, 0)
            AS hist_ctr,

        1.0 * outcome_impressions
            / NULLIF(outcome_days, 0)
            AS outcome_imp_per_day

    FROM aggregated

    WHERE hist_days >= 14
      AND outcome_days >= 7
      AND hist_impressions >= 50
)

SELECT
    client_hash_id,
    content_hash_id,
    hist_days,
    outcome_days,
    hist_imp_per_day,
    hist_click_per_day,
    hist_ctr,
    hist_avg_position,
    hist_position_sd,

    CAST(
        outcome_imp_per_day
        < 0.8 * hist_imp_per_day
        AS INTEGER
    ) AS is_declining_proxy

FROM framed
"""

raw_frame = con.sql(feature_sql).df()

FEATURES = [
    "hist_imp_per_day",
    "hist_click_per_day",
    "hist_ctr",
    "hist_avg_position",
    "hist_position_sd",
]

LABEL = "is_declining_proxy"
GROUP = "client_hash_id"

model_frame = (
    raw_frame
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=FEATURES
        + [LABEL, GROUP, "content_hash_id"]
    )
    .copy()
)

model_frame[LABEL] = model_frame[LABEL].astype(int)

assert not model_frame.empty
assert model_frame[LABEL].nunique() == 2
assert (
    model_frame.duplicated(
        [GROUP, "content_hash_id"]
    ).sum()
    == 0
)

data_checks = pd.DataFrame([
    {
        "check": "Eligible rows",
        "observed": len(model_frame),
        "week4_reference": 2348,
        "matches": len(model_frame) == 2348,
    },
    {
        "check": "Positive base rate",
        "observed": model_frame[LABEL].mean(),
        "week4_reference": 0.2355,
        "matches": np.isclose(
            model_frame[LABEL].mean(),
            0.2355,
            atol=0.0001,
        ),
    },
    {
        "check": "Honest feature count",
        "observed": len(FEATURES),
        "week4_reference": 5,
        "matches": len(FEATURES) == 5,
    },
])

display(data_checks.round(4))

print("Pseudonymized clients:", model_frame[GROUP].nunique())
print("Duplicate client-content rows: 0")
print("June/_sample queried: NO")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,check,observed,week4_reference,matches
0,Eligible rows,2348.0000,2348.0000,True
1,Positive base rate,0.2355,0.2355,True
2,Honest feature count,5.0000,5.0000,True


Pseudonymized clients: 14
Duplicate client-content rows: 0
June/_sample queried: NO


## 2. Split design

I use GroupShuffleSplit with 25% of pseudonymized clients held out and random_state=42. Every page belonging to one client remains entirely in either training or test data.

This prevents client-specific traffic scale and content patterns from appearing in both partitions. The Week-4 rule thresholds, imputer, scaler, and learned models are fitted only on training clients. The baseline and both learned models are then evaluated on exactly the same held-out client rows.

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED,
)

train_idx, test_idx = next(
    splitter.split(
        model_frame[FEATURES],
        model_frame[LABEL],
        groups=model_frame[GROUP],
    )
)

train_frame = (
    model_frame
    .iloc[train_idx]
    .reset_index(drop=True)
)

test_frame = (
    model_frame
    .iloc[test_idx]
    .reset_index(drop=True)
)

train_clients = set(train_frame[GROUP])
test_clients = set(test_frame[GROUP])

assert train_clients.isdisjoint(test_clients)
assert train_frame[LABEL].nunique() == 2
assert test_frame[LABEL].nunique() == 2
assert len(train_frame) + len(test_frame) == len(model_frame)

split_summary = pd.DataFrame([
    {
        "partition": "train",
        "rows": len(train_frame),
        "clients": train_frame[GROUP].nunique(),
        "positive_rate": train_frame[LABEL].mean(),
    },
    {
        "partition": "test",
        "rows": len(test_frame),
        "clients": test_frame[GROUP].nunique(),
        "positive_rate": test_frame[LABEL].mean(),
    },
])

display(split_summary.round(4))

print(
    "Client overlap:",
    len(train_clients.intersection(test_clients))
)
print("Split random seed:", SEED)

# ---------------------------------------------------------
# Recreate Week-4 rule using training thresholds only
# ---------------------------------------------------------

baseline_thresholds = {
    "high_demand_q75": float(
        train_frame["hist_imp_per_day"].quantile(0.75)
    ),
    "low_ctr_q25": float(
        train_frame["hist_ctr"].quantile(0.25)
    ),
    "high_volatility_q75": float(
        train_frame["hist_position_sd"].quantile(0.75)
    ),
}

def add_week4_rule_score(frame):
    scored = frame.copy()

    scored["flag_high_demand"] = (
        scored["hist_imp_per_day"]
        >= baseline_thresholds["high_demand_q75"]
    ).astype(int)

    scored["flag_position_opportunity"] = (
        scored["hist_avg_position"].between(
            4.0,
            20.0,
            inclusive="both",
        )
    ).astype(int)

    scored["flag_low_ctr"] = (
        scored["hist_ctr"]
        <= baseline_thresholds["low_ctr_q25"]
    ).astype(int)

    scored["flag_position_instability"] = (
        scored["hist_position_sd"]
        >= baseline_thresholds["high_volatility_q75"]
    ).astype(int)

    scored["score_week4_rule"] = (
        4 * scored["flag_high_demand"]
        + 3 * scored["flag_position_opportunity"]
        + 2 * scored["flag_low_ctr"]
        + 1 * scored["flag_position_instability"]
    )

    return scored

test_scored = add_week4_rule_score(test_frame)

threshold_table = pd.DataFrame({
    "training_only_threshold": list(
        baseline_thresholds.keys()
    ),
    "value": list(baseline_thresholds.values()),
})

display(threshold_table.round(6))

,partition,rows,clients,positive_rate
0,train,2120,10,0.2547
1,test,228,4,0.0570


Client overlap: 0
Split random seed: 42


,training_only_threshold,value
0,high_demand_q75,592.925000
1,low_ctr_q25,0.002917
2,high_volatility_q75,5.004918


## 3. Train + compare vs my baseline

I compare the original Week-4 scoring rule, Logistic Regression, and Random Forest on exactly the same held-out clients. All learned models use only the same five historical features.

Precision@20 is the primary metric because only twenty pages can enter the review queue. I additionally report Precision@50, Lift@20, Average Precision, and ROC-AUC. Random Forest is preferred only if it improves Precision@20 over Logistic Regression by at least 0.05; otherwise the simpler model remains preferred.

In [ ]:
X_train = train_frame[FEATURES]
y_train = train_frame[LABEL]

X_test = test_frame[FEATURES]
y_test = test_frame[LABEL]

models = {
    "Logistic Regression": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=SEED,
            ),
        ),
    ]),

    "Random Forest": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=400,
                max_depth=6,
                min_samples_leaf=10,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    ]),
}

score_columns = {
    "Week-4 rule (grouped holdout)":
        "score_week4_rule",

    "Logistic Regression":
        "score_logistic",

    "Random Forest":
        "score_random_forest",
}

evaluation = test_scored.copy()
fitted_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)

    fitted_models[model_name] = model

    evaluation[score_columns[model_name]] = (
        model.predict_proba(X_test)[:, 1]
    )

def rank_for_review(frame, score_column):
    return (
        frame
        .sort_values(
            by=[
                score_column,
                "hist_imp_per_day",
                "hist_click_per_day",
                "content_hash_id",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

def precision_at_k(frame, score_column, k):
    ranked = rank_for_review(
        frame,
        score_column,
    )

    effective_k = min(k, len(ranked))

    return float(
        ranked
        .head(effective_k)[LABEL]
        .mean()
    )

def calculate_metrics(method, score_column):
    scores = evaluation[score_column]
    base_rate = float(evaluation[LABEL].mean())

    p20 = precision_at_k(
        evaluation,
        score_column,
        20,
    )

    p50 = precision_at_k(
        evaluation,
        score_column,
        50,
    )

    return {
        "method": method,
        "train_rows": len(train_frame),
        "test_rows": len(test_frame),
        "test_clients": test_frame[GROUP].nunique(),
        "test_base_rate": base_rate,
        "precision_at_20": p20,
        "precision_at_50": p50,
        "lift_at_20": (
            p20 / base_rate
            if base_rate > 0
            else np.nan
        ),
        "average_precision": (
            average_precision_score(
                y_test,
                scores,
            )
        ),
        "roc_auc": roc_auc_score(
            y_test,
            scores,
        ),
    }

method_order = [
    "Week-4 rule (grouped holdout)",
    "Logistic Regression",
    "Random Forest",
]

comparison = pd.DataFrame([
    calculate_metrics(
        method,
        score_columns[method],
    )
    for method in method_order
])

baseline_p20 = float(
    comparison.loc[
        comparison["method"]
        == "Week-4 rule (grouped holdout)",
        "precision_at_20",
    ].iloc[0]
)

comparison["delta_p20_vs_baseline"] = (
    comparison["precision_at_20"]
    - baseline_p20
)

logistic_p20 = float(
    comparison.loc[
        comparison["method"]
        == "Logistic Regression",
        "precision_at_20",
    ].iloc[0]
)

forest_p20 = float(
    comparison.loc[
        comparison["method"]
        == "Random Forest",
        "precision_at_20",
    ].iloc[0]
)

if forest_p20 >= logistic_p20 + 0.05:
    selected_model_name = "Random Forest"

    selection_reason = (
        "Random Forest earned its additional complexity "
        "by improving Precision@20 by at least 0.05."
    )
else:
    selected_model_name = "Logistic Regression"

    selection_reason = (
        "Random Forest did not improve Precision@20 by "
        "the required 0.05, so the simpler Logistic "
        "Regression remains preferred."
    )

selected_score_column = score_columns[
    selected_model_name
]

print("FINAL MODEL-VS-BASELINE TABLE")
display(comparison.round(4))

historical_reference = pd.DataFrame([
    {
        "result":
            "Week-4 complete-slice result",
        "precision_at_20": 0.35,
        "held_out_clients": False,
        "directly_comparable": False,
    },
    {
        "result":
            "Week-4 rule on Week-5 holdout",
        "precision_at_20": baseline_p20,
        "held_out_clients": True,
        "directly_comparable": True,
    },
])

print("WEEK-4 HISTORICAL REFERENCE")
display(historical_reference.round(4))

print("Preferred learned model:", selected_model_name)
print("Reason:", selection_reason)

reproducibility = pd.DataFrame([{
    "random_seed": SEED,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "duckdb": duckdb.__version__,
}])

display(reproducibility)

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    OUTPUT_DIR / "w05_model_comparison.csv",
    index=False,
)

assert comparison["test_rows"].nunique() == 1
assert comparison["test_clients"].nunique() == 1
assert comparison["test_base_rate"].nunique() == 1
assert comparison["precision_at_20"].between(0, 1).all()

FINAL MODEL-VS-BASELINE TABLE


,method,train_rows,test_rows,test_clients,test_base_rate,precision_at_20,precision_at_50,lift_at_20,average_precision,roc_auc,delta_p20_vs_baseline
0,Week-4 rule (grouped holdout),2120,228,4,0.057,0.0,0.04,0.0,0.0478,0.3955,0.0
1,Logistic Regression,2120,228,4,0.057,0.0,0.04,0.0,0.0461,0.3284,0.0
2,Random Forest,2120,228,4,0.057,0.0,0.02,0.0,0.0591,0.5120,0.0


WEEK-4 HISTORICAL REFERENCE


,result,precision_at_20,held_out_clients,directly_comparable
0,Week-4 complete-slice result,0.35,False,False
1,Week-4 rule on Week-5 holdout,0.00,True,True


Preferred learned model: Logistic Regression
Reason: Random Forest did not improve Precision@20 by the required 0.05, so the simpler Logistic Regression remains preferred.


,random_seed,numpy,pandas,scikit_learn,duckdb
0,42,2.0.2,2.2.2,1.6.1,1.3.2


## 4. Errors and interpretation

For this ranking task, a Top-20 false positive is a selected page whose observed decline proxy is 0. A positive outside Top-20 is a declining page missed because it ranked below the fixed review capacity.

I inspect three pseudonymized wrong cases, error size across demand quartiles, held-out permutation importance, and Logistic Regression coefficient directions. Feature importance represents predictive association and does not prove that changing a feature would cause traffic recovery.

In [ ]:
ranked_selected = rank_for_review(
    evaluation,
    selected_score_column,
)

ranked_selected.insert(
    0,
    "model_rank",
    np.arange(
        1,
        len(ranked_selected) + 1,
    ),
)

ranked_selected["selected_top20"] = (
    ranked_selected["model_rank"] <= 20
)

ranked_selected["error_type"] = np.select(
    [
        (
            ranked_selected["selected_top20"]
            & (ranked_selected[LABEL] == 0)
        ),
        (
            ~ranked_selected["selected_top20"]
            & (ranked_selected[LABEL] == 1)
        ),
    ],
    [
        "top20_false_positive",
        "positive_outside_top20",
    ],
    default="correct_for_top20_decision",
)

ranked_selected["absolute_probability_error"] = (
    np.abs(
        ranked_selected[LABEL]
        - ranked_selected[selected_score_column]
    )
)

top20 = ranked_selected[
    ranked_selected["selected_top20"]
].copy()

false_positives = ranked_selected[
    ranked_selected["error_type"]
    == "top20_false_positive"
].copy()

missed_positives = ranked_selected[
    ranked_selected["error_type"]
    == "positive_outside_top20"
].copy()

error_summary = pd.DataFrame([{
    "preferred_model": selected_model_name,
    "top20_true_positives": int(
        top20[LABEL].sum()
    ),
    "top20_false_positives": int(
        (top20[LABEL] == 0).sum()
    ),
    "positive_pages_outside_top20":
        len(missed_positives),
    "total_test_positives": int(
        ranked_selected[LABEL].sum()
    ),
}])

print("TOP-20 ERROR SUMMARY")
display(error_summary)

# Select two false positives and one missed positive.
wrong_cases = pd.concat([
    false_positives.head(2),
    missed_positives.head(
        max(
            0,
            3 - min(2, len(false_positives)),
        )
    ),
])

# Fallback when fewer than three operational errors exist.
if len(wrong_cases) < 3:
    remaining = ranked_selected.loc[
        ~ranked_selected.index.isin(
            wrong_cases.index
        )
    ].nlargest(
        3 - len(wrong_cases),
        "absolute_probability_error",
    )

    wrong_cases = pd.concat([
        wrong_cases,
        remaining,
    ])

wrong_cases = wrong_cases.head(3).copy()

def explain_wrong_case(row):
    if row["error_type"] == "top20_false_positive":
        return (
            "Historical signals produced a high review "
            "score, but this page did not meet the short "
            "future-decline proxy. Temporary demand, "
            "seasonality, or client-specific patterns may "
            "make the page appear riskier."
        )

    if row["error_type"] == "positive_outside_top20":
        return (
            "The page met the future-decline proxy but "
            "ranked below the fixed 20-page review "
            "capacity because its historical signals "
            "appeared less risky."
        )

    return (
        "This row has a large probability error even "
        "though it is not one of the two primary "
        "Top-20 error types."
    )

wrong_cases["why_this_case_is_hard"] = (
    wrong_cases.apply(
        explain_wrong_case,
        axis=1,
    )
)

wrong_case_columns = [
    "model_rank",
    GROUP,
    "content_hash_id",
    LABEL,
    selected_score_column,
    "error_type",
    "hist_imp_per_day",
    "hist_click_per_day",
    "hist_ctr",
    "hist_avg_position",
    "hist_position_sd",
    "why_this_case_is_hard",
]

print("THREE PSEUDONYMIZED WRONG CASES")
display(
    wrong_cases[
        wrong_case_columns
    ].round(5)
)

# ---------------------------------------------------------
# Error size by demand quartile
# ---------------------------------------------------------

ranked_selected["demand_quartile"] = pd.qcut(
    ranked_selected[
        "hist_imp_per_day"
    ].rank(method="first"),
    q=4,
    labels=[
        "Q1 lowest",
        "Q2",
        "Q3",
        "Q4 highest",
    ],
)

error_by_demand = (
    ranked_selected
    .groupby(
        "demand_quartile",
        observed=True,
    )
    .agg(
        rows=(LABEL, "size"),
        observed_positive_rate=(
            LABEL,
            "mean",
        ),
        mean_predicted_risk=(
            selected_score_column,
            "mean",
        ),
        mean_absolute_error=(
            "absolute_probability_error",
            "mean",
        ),
    )
    .reset_index()
)

print("ERRORS BY HISTORICAL-DEMAND QUARTILE")
display(error_by_demand.round(4))

# ---------------------------------------------------------
# Held-out permutation importance
# ---------------------------------------------------------

selected_model = fitted_models[
    selected_model_name
]

permutation = permutation_importance(
    selected_model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=SEED,
    scoring="average_precision",
    n_jobs=-1,
)

importance_table = (
    pd.DataFrame({
        "feature": FEATURES,
        "importance_mean":
            permutation.importances_mean,
        "importance_sd":
            permutation.importances_std,
    })
    .sort_values(
        "importance_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("HELD-OUT PERMUTATION IMPORTANCE")
display(importance_table.round(5))

# ---------------------------------------------------------
# Logistic Regression coefficient directions
# ---------------------------------------------------------

logistic_coefficients = pd.DataFrame({
    "feature": FEATURES,

    "standardized_coefficient":
        fitted_models[
            "Logistic Regression"
        ].named_steps["model"].coef_[0],
})

logistic_coefficients["direction"] = np.where(
    logistic_coefficients[
        "standardized_coefficient"
    ] >= 0,

    "higher value -> higher estimated decline risk",

    "higher value -> lower estimated decline risk",
)

logistic_coefficients = (
    logistic_coefficients
    .assign(
        absolute_coefficient=lambda x:
            x["standardized_coefficient"].abs()
    )
    .sort_values(
        "absolute_coefficient",
        ascending=False,
    )
    .drop(columns="absolute_coefficient")
    .reset_index(drop=True)
)

print("LOGISTIC REGRESSION DIRECTIONS")
display(logistic_coefficients.round(5))

# ---------------------------------------------------------
# Automatic result-based conclusion
# ---------------------------------------------------------

selected_p20 = precision_at_k(
    evaluation,
    selected_score_column,
    20,
)

if selected_p20 > baseline_p20:
    comparison_sentence = (
        f"The preferred learned model improved held-out "
        f"Precision@20 from {baseline_p20:.3f} "
        f"to {selected_p20:.3f}."
    )

elif np.isclose(selected_p20, baseline_p20):
    comparison_sentence = (
        f"The preferred learned model tied the baseline "
        f"at Precision@20 = {selected_p20:.3f}."
    )

else:
    comparison_sentence = (
        f"The preferred learned model achieved "
        f"Precision@20 = {selected_p20:.3f}, below the "
        f"baseline result of {baseline_p20:.3f}. The "
        f"baseline remains stronger on this holdout."
    )

hardest_group = (
    error_by_demand
    .sort_values(
        "mean_absolute_error",
        ascending=False,
    )
    .iloc[0]
)

conclusion = f"""
### Data-dependent conclusion

{comparison_sentence}

The preferred model's Top-20 contained
**{int(top20[LABEL].sum())} proxy-positive selections**
and **{int((top20[LABEL] == 0).sum())} false positives**.
A further **{len(missed_positives)} proxy-positive pages**
ranked outside the fixed Top-20 review capacity.

The largest mean absolute probability error appeared in
**{hardest_group['demand_quartile']}**. The strongest held-out
permutation feature was
**`{importance_table.iloc[0]['feature']}`**. This is a predictive
association and not evidence that changing this feature would
cause search-traffic recovery.

This result is limited to one eligible March 2026 slice, one
client-grouped holdout, and a short ten-day outcome window.
Temporary demand, seasonality, measurement noise, and the
availability filter can affect the observed proxy. The model
should therefore support human review rather than make an
automatic refresh decision.
"""

display(Markdown(conclusion))

assert len(wrong_cases) == 3
assert len(importance_table) == len(FEATURES)
assert LABEL not in FEATURES
assert GROUP not in FEATURES
assert "content_hash_id" not in FEATURES

print("PASS: error analysis completed.")
print("PASS: feature interpretation completed.")
print("PASS: no future or ID field is a model feature.")

TOP-20 ERROR SUMMARY


,preferred_model,top20_true_positives,top20_false_positives,positive_pages_outside_top20,total_test_positives
0,Logistic Regression,0,20,13,13


THREE PSEUDONYMIZED WRONG CASES


,model_rank,client_hash_id,content_hash_id,is_declining_proxy,score_logistic,error_type,hist_imp_per_day,hist_click_per_day,hist_ctr,hist_avg_position,hist_position_sd,why_this_case_is_hard
0,1,client_fef1a8f436438636,content_0aaa197051f58d6f,0,0.85790,top20_false_positive,2011.31250,1.0625,0.00053,34.73665,5.07147,"Historical signals produced a high review score, but this page did not meet the short future-decline proxy. Temporary demand, seasonalit..."
1,2,client_fef1a8f436438636,content_7960401fa9cfe53b,0,0.78324,top20_false_positive,977.81250,6.8750,0.00703,37.14976,8.75657,"Historical signals produced a high review score, but this page did not meet the short future-decline proxy. Temporary demand, seasonalit..."
20,21,client_fef1a8f436438636,content_ff523522181a94f9,1,0.47339,positive_outside_top20,116.93333,0.8000,0.00684,17.40764,2.58738,The page met the future-decline proxy but ranked below the fixed 20-page review capacity because its historical signals appeared less ri...


ERRORS BY HISTORICAL-DEMAND QUARTILE


...

,feature,importance_mean,importance_sd
0,hist_click_per_day,0.00025,0.00107
1,hist_imp_per_day,-0.00368,0.00334
2,hist_position_sd,-0.00608,0.00199
3,hist_ctr,-0.00726,0.00209
4,hist_avg_position,-0.01141,0.02965


LOGISTIC REGRESSION DIRECTIONS


,feature,standardized_coefficient,direction
0,hist_avg_position,0.91376,higher value -> higher estimated decline risk
1,hist_imp_per_day,0.15807,higher value -> higher estimated decline risk
2,hist_position_sd,-0.13764,higher value -> lower estimated decline risk
3,hist_ctr,-0.12948,higher value -> lower estimated decline risk
4,hist_click_per_day,-0.09911,higher value -> lower estimated decline risk



### Data-dependent conclusion

The preferred learned model tied the baseline at Precision@20 = 0.000.

The preferred model's Top-20 contained
**0 proxy-positive selections**
and **20 false positives**.
A further **13 proxy-positive pages**
ranked outside the fixed Top-20 review capacity.

The largest mean absolute probability error appeared in
**Q4 highest**. The strongest held-out
permutation feature was
**`hist_click_per_day`**. This is a predictive
association and not evidence that changing this feature would
cause search-traffic recovery.

This result is limited to one eligible March 2026 slice, one
client-grouped holdout, and a short ten-day outcome window.
Temporary demand, seasonality, measurement noise, and the
availability filter can affect the observed proxy. The model
should therefore support human review rather than make an
automatic refresh decision.


PASS: error analysis completed.
PASS: feature interpretation completed.
PASS: no future or ID field is a model feature.


## Self-check

- [x] Every section contains explanation and supporting code.
- [x] The notebook runs top to bottom without errors.
- [x] The current run reproduces 2,348 eligible rows.
- [x] The positive base rate is approximately 0.2355.
- [x] Exactly five historical features are used.
- [x] Training and test clients have zero overlap.
- [x] The baseline and models use the same held-out rows.
- [x] The final table reports Precision@20 and base rate.
- [x] Three wrong cases are displayed and interpreted.
- [x] Permutation importance is calculated on held-out data.
- [x] Logistic Regression coefficient directions are shown.
- [x] IDs and future measurements are excluded from features.
- [x] June 2026 and the `_sample` table were not queried.
- [x] No token, client name, URL, or private query appears.
- [x] Claims use observed and decision-support language.
- [x] I read the research paper linked on the assignment card.
- [x] Saved and committed as `work/notebooks/w05_model.ipynb`.